# The nano frame grid on a rented card

Trains a grid of nano Beatrix crafts (Castor, the rotary softmax body; Pollux, the alternating body; both with the byte-class table beside the aleph address) on dynamically framed rows, one run after another on this card, and ships every finished run (its record and its weights) to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training).

Before running: put a Hugging Face token with write access to that repo in this notebook's secrets as `HF_TOKEN` (the key icon on the left). Then run the cells in order. A grid resumes where it stopped if the session restarts (`runner_done.txt` in the output folder); a file named `STOP` in the output folder ends the grid between runs.

In [ ]:
# 1. the trainer at a tag (pulls the aleph library at its pinned commit, torch 2.11 cu128, numpy, safetensors, huggingface_hub, datasets)
!pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.9"
import geolip.nano_beatrix, torch
print("trainer", geolip.nano_beatrix.__version__, "| torch", torch.__version__, "| card", torch.cuda.get_device_name(0))

In [ ]:
# 2. the token (never printed) so each finished run ships to the Hub
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# 3. the training text and the held-out file from fineweb-edu (tokenless; about 160 MB; a few minutes)
OUT = "/content/out"
!python -m geolip.nano_beatrix.bed_corpus {OUT}/bed 160e6

In [ ]:
# 4. the grid. C1 = the register grid at 4,096: two bodies x two seeds x three registers = 12 runs; every finished run ships.
#    Comma lists expand into the product (arms, seeds, register, ask, facts, frame_records, nest, overlap, ctx, steps, mb, plant, tags);
#    --sizes keeps its own list. Each run prints its job size, the time spent and the time left; the runner prints the grid's place.
ARMS = "castor-both,pollux-both"
SEEDS = "1,2"
REGISTERS = "caption,math,code"
CTX, MB, STEPS = 4096, 8, 12000
SIZES = "32,128,256,512"
PUSH = "AbstractPhil/alephllm-nano-beatrix-training"
!python -m geolip.nano_beatrix.bed_runner --out={OUT} --arms={ARMS} --seeds={SEEDS} --register={REGISTERS} --ctx={CTX} --mb={MB} --steps={STEPS} \
    --frames=dyn --plant=1.0 --sizes={SIZES} --frame_records=4 --nest=0.3 --overlap=0.0 --ask=any --tags=mode --log_every=100 --push={PUSH}

## The other grids

Change the variables in cell 4 and run it again; finished runs are skipped.

| grid | variables |
|---|---|
| C2 the in-window capacity axis | `SEEDS = "1"`, `REGISTERS = "caption"`, add `--frame_records=4,8,16` in place of `--frame_records=4`, `SIZES = "32,128,256,512,1024"` |
| C3 the weights source | `SEEDS = "1"`, `REGISTERS = "caption"`, add `--facts=64,256,1024` and set `--ask=weights` |
| C4 the scrutiny set | `--overlap=0.3 --nest=0.6` and `--ask=nested,overlap` |

A smaller card: `MB = 4` at 4,096 (the step count stays; the run takes longer). A quick check that everything works before a long grid: add `--steps=200` and `--seeds=1` with one register.